# Batch Process Fault Bucketing & Extraction Pipeline

This notebook allows you to:
1. View all folders under `data/input/12-05-26-aarya/fault-bucketing/`
2. Select one or multiple folders to process (each folder contains `traces/raw_trace.json`)
3. Run the fault bucketing and extraction pipeline on the selected folders
4. Save results to `data/output/12-05-26-aarya/` retaining folder structure

## 1. Import Required Libraries

In [1]:
import asyncio
import json
import sys
from pathlib import Path
from typing import List, Dict, Any
import pandas as pd
from datetime import datetime

# Import the pipeline function
from run_bucketing_and_extraction_pipeline import run_pipeline

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 2. Define Configuration Parameters

In [2]:
# Base directories
BASE_DIR = Path(r"C:\Users\meemankgupta\Music\Project\infosys\certifier")
INPUT_BASE_DIR  = BASE_DIR / "data" / "input"  / "21-05-ravi-top10" / "1960bc89-361a-4fcb-af86-699113f09ec9-langfuse" / "e8650990-3905-4e56-b458-1ddb84d3972e" / "fault-bucketing"
OUTPUT_BASE_DIR = BASE_DIR / "data" / "output" / "26-05-ravi-top10"

# Load configuration
try:
    from utils.load_config import ConfigLoader
    config = ConfigLoader.load_config()
    fault_config = config.get("fault_bucketing_config", {})
    pipeline_config = fault_config.get("pipeline", {})
    classifier_config = fault_config.get("classifier", {})
    BATCH_SIZE = pipeline_config.get("default_batch_size", 1)
    FAULT_PRUNING = classifier_config.get("fault_pruning", True)
    CACHE_ENABLED = classifier_config.get("cache_enabled", True)
    INCLUDE_EVENT_INPUT = classifier_config.get("include_event_input", False)
    PROMPT_PATH = classifier_config.get("prompt_path", None)
    STORE_TO_MONGODB = False
    print("✓ Configuration loaded successfully")
    print(f"\nPipeline Configuration:")
    print(f"  Batch Size: {BATCH_SIZE}")
    print(f"  Fault Pruning: {FAULT_PRUNING}")
    print(f"  Cache Enabled: {CACHE_ENABLED}")
    print(f"  Include Event Input: {INCLUDE_EVENT_INPUT}")
    print(f"  Store to MongoDB: {STORE_TO_MONGODB}")
except Exception as e:
    print(f"⚠ Could not load config: {e}")
    BATCH_SIZE = 1
    STORE_TO_MONGODB = False
    FAULT_PRUNING = True
    CACHE_ENABLED = True
    INCLUDE_EVENT_INPUT = False
    PROMPT_PATH = None

print(f"\nDirectories:")
print(f"  Input directory : {INPUT_BASE_DIR}")
print(f"  Output directory: {OUTPUT_BASE_DIR}")
print(f"  Input exists    : {INPUT_BASE_DIR.exists()}")

✓ Configuration loaded successfully

Pipeline Configuration:
  Batch Size: 1
  Fault Pruning: True
  Cache Enabled: True
  Include Event Input: False
  Store to MongoDB: False

Directories:
  Input directory : C:\Users\meemankgupta\Music\Project\infosys\certifier\data\input\21-05-ravi-top10\1960bc89-361a-4fcb-af86-699113f09ec9-langfuse\e8650990-3905-4e56-b458-1ddb84d3972e\fault-bucketing
  Output directory: C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10
  Input exists    : True


## 3. Define Helper Functions

Function to scan directories and find folders containing `traces/raw_trace.json` files.

In [3]:
def find_trace_files(base_dir: Path) -> List[Dict[str, Any]]:
    """
    Find all raw_trace.json files. folder_id (run UUID) is already unique —
    no index prefix needed. Marks runs with existing metrics as 'done' so
    the processing loop can skip them.
    """
    trace_files = []
    for folder in sorted(base_dir.iterdir(), key=lambda p: p.name):
        if folder.is_dir():
            trace_path = folder / "traces" / "raw_trace.json"
            if trace_path.exists():
                folder_id = folder.name
                output_path = OUTPUT_BASE_DIR / folder_id
                # Check if this run was already extracted (has at least one metrics file)
                existing_metrics = list(output_path.rglob("*_metrics.json"))
                status = "done" if existing_metrics else "pending"
                trace_files.append({
                    "folder_id": folder_id,
                    "trace_path": str(trace_path),
                    "output_path": str(output_path),
                    "status": status,
                    "existing_metrics": len(existing_metrics),
                })
    return trace_files

print("✓ Utility functions defined")

✓ Utility functions defined


## 4. Select Folders to Process

View available folders (each contains `traces/raw_trace.json`) and select which ones to process.

In [4]:
# Discover all folders containing raw_trace.json
all_trace_files = find_trace_files(INPUT_BASE_DIR)
print(f"Found {len(all_trace_files)} folders with raw_trace.json files\n")

print("AVAILABLE FOLDERS:")
print("=" * 100)
for i, trace_info in enumerate(all_trace_files):
    print(f"  [{i:2d}]   {trace_info['folder_id']}")
print("=" * 100)

# Process ALL folders
selected_indices = list(range(0, len(all_trace_files)))
selected_trace_files = [all_trace_files[i] for i in selected_indices]

print(f"\n✓ SELECTED {len(selected_trace_files)} FOLDER(S) TO PROCESS:")
print("=" * 100)
for i, trace_info in enumerate(selected_trace_files):
    print(f"  [{selected_indices[i]}] {trace_info['folder_id']}")
    print(f"      → {trace_info['trace_path']}")
print("=" * 100)

Found 10 folders with raw_trace.json files

AVAILABLE FOLDERS:
  [ 0]   000098ae-eac9-4042-ac11-546006172858
  [ 1]   55363bdb-3479-4bca-989d-b60b8afeacd8
  [ 2]   71f3c167-5202-4750-9de4-0b4ca44e726b
  [ 3]   a3d3785f-7521-4b73-85d4-e0e5c5a54603
  [ 4]   a599d630-f6a7-4269-b300-c7b87242c1aa
  [ 5]   a9122c3d-4cd3-4a0b-bf99-b8151d7c1170
  [ 6]   ada79287-1eb5-4e6b-8611-cdc21af9e263
  [ 7]   b1bf3fb2-39ad-49b3-abf0-240d7ed2be20
  [ 8]   c52aa625-cd95-43e6-9f6f-7ddf1def07ef
  [ 9]   da495936-aa5c-4fb1-a0a8-7cf5e2922222

✓ SELECTED 10 FOLDER(S) TO PROCESS:
  [0] 000098ae-eac9-4042-ac11-546006172858
      → C:\Users\meemankgupta\Music\Project\infosys\certifier\data\input\21-05-ravi-top10\1960bc89-361a-4fcb-af86-699113f09ec9-langfuse\e8650990-3905-4e56-b458-1ddb84d3972e\fault-bucketing\000098ae-eac9-4042-ac11-546006172858\traces\raw_trace.json
  [1] 55363bdb-3479-4bca-989d-b60b8afeacd8
      → C:\Users\meemankgupta\Music\Project\infosys\certifier\data\input\21-05-ravi-top10\1960bc89-361a-4f

## 5. Process Selected Folders Sequentially

Process each selected folder's `raw_trace.json` file **one at a time**. Each trace is fully processed and output saved before moving to the next.

In [5]:
results_summary = []
start_time = datetime.now()

print(f"Starting sequential processing of {len(selected_trace_files)} folder(s) at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

for idx, trace_info in enumerate(selected_trace_files, 1):
    folder_id   = trace_info["folder_id"]
    trace_path  = trace_info["trace_path"]
    output_path = trace_info["output_path"]   # OUTPUT_BASE_DIR / folder_id  (UUID, already unique)

    print(f"\n[{idx}/{len(selected_trace_files)}] Folder: {folder_id}")
    print(f"  → Input : {trace_path}")
    print(f"  → Output: {output_path}")

    # Skip runs that already completed to avoid deleting their fault_buckets
    if trace_info.get("status") == "done":
        print(f"  ⏭  SKIPPED — {trace_info['existing_metrics']} metrics file(s) already exist")
        results_summary.append({
            "folder_id": folder_id, "output_path": output_path,
            "status": "skipped", "faults_extracted": trace_info["existing_metrics"],
            "total_tokens": 0, "error": None,
        })
        continue

    try:
        results = await run_pipeline(
            trace_file=trace_path,
            output_dir=output_path,
            batch_size=BATCH_SIZE,
            store_to_mongodb=STORE_TO_MONGODB,
            fault_pruning=FAULT_PRUNING,
            cache_enabled=CACHE_ENABLED,
            include_event_input=INCLUDE_EVENT_INPUT,
            prompt_path=PROMPT_PATH,
        )

        total_tokens = sum(r["token_usage"]["total_tokens"] for r in results)
        results_summary.append({
            "folder_id": folder_id, "output_path": output_path,
            "status": "success", "faults_extracted": len(results),
            "total_tokens": total_tokens, "error": None,
        })
        print(f"  ✓ SUCCESS: {len(results)} faults extracted, {total_tokens:,} tokens used")

    except Exception as exc:
        results_summary.append({
            "folder_id": folder_id, "output_path": output_path,
            "status": "failed", "faults_extracted": 0,
            "total_tokens": 0, "error": str(exc),
        })
        print(f"  ✗ FAILED: {exc}")

duration = datetime.now() - start_time
print("\n" + "=" * 80)
print(f"Done in {duration}. Success: {sum(1 for r in results_summary if r['status'] == 'success')}, "
      f"Skipped: {sum(1 for r in results_summary if r['status'] == 'skipped')}, "
      f"Failed: {sum(1 for r in results_summary if r['status'] == 'failed')}")

2026-05-26 22:06:43,845 - [run_bucketing_and_extraction_pipeline.py : run_pipeline : 99] - INFO - ============================================================
2026-05-26 22:06:43,846 - [run_bucketing_and_extraction_pipeline.py : run_pipeline : 100] - INFO - STEP 1: Fault Bucketing
2026-05-26 22:06:43,847 - [run_bucketing_and_extraction_pipeline.py : run_pipeline : 101] - INFO - ============================================================
2026-05-26 22:06:43,897 - [fault_bucketing.py : _load_trace : 499] - INFO - Loaded 82 events from raw_trace.json
2026-05-26 22:06:43,899 - [fault_bucketing.py : _extract_tokens_from_trace : 562] - INFO - Extracted run-level tokens from trace: input=189725, output=54898
2026-05-26 22:06:43,900 - [fault_bucketing.py : _extract_agent_metadata : 229] - INFO - Agent metadata extracted: id=1960bc89-361a-4fcb-af86-699113f09ec9, name=demoinfra2, version=3.0.0, experiment_id=e8650990-3905-4e56-b458-1ddb84d3972e, run_id=000098ae-eac9-4042-ac11-546006172858
2026-

Starting sequential processing of 10 folder(s) at 2026-05-26 22:06:43

[1/10] Folder: 000098ae-eac9-4042-ac11-546006172858
  → Input : C:\Users\meemankgupta\Music\Project\infosys\certifier\data\input\21-05-ravi-top10\1960bc89-361a-4fcb-af86-699113f09ec9-langfuse\e8650990-3905-4e56-b458-1ddb84d3972e\fault-bucketing\000098ae-eac9-4042-ac11-546006172858\traces\raw_trace.json
  → Output: C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\000098ae-eac9-4042-ac11-546006172858


2026-05-26 22:06:44,414 - [azure_openai_util.py : get_clients : 103] - INFO - Created Azure OpenAI client for model: embedding_model
2026-05-26 22:06:44,662 - [azure_openai_util.py : get_clients : 103] - INFO - Created Azure OpenAI client for model: gpt-4o
2026-05-26 22:06:44,916 - [azure_openai_util.py : get_clients : 103] - INFO - Created Azure OpenAI client for model: gpt-5.2
2026-05-26 22:06:44,917 - [azure_openai_util.py : __init__ : 46] - INFO - AzureLLMClient initialized successfully
2026-05-26 22:06:44,920 - [azure_openai_util.py : _get_or_create_agent : 137] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-26 22:06:49,741 - [fault_bucketing.py : run : 1178] - INFO - Batch 32/79 processed (1 events): fault_span_skips=0, scaffolding_skips=0, deterministic_assignments=0, zero_candidate_skips=0, llm_classifications=1, tokens=7292/381
2026-05-26 22:06:49,744 - [azure_openai_util.py : _get_or_create_agent : 137] - WARNING - No specifi

  ✓ SUCCESS: 3 faults extracted, 2,270,781 tokens used

[2/10] Folder: 55363bdb-3479-4bca-989d-b60b8afeacd8
  → Input : C:\Users\meemankgupta\Music\Project\infosys\certifier\data\input\21-05-ravi-top10\1960bc89-361a-4fcb-af86-699113f09ec9-langfuse\e8650990-3905-4e56-b458-1ddb84d3972e\fault-bucketing\55363bdb-3479-4bca-989d-b60b8afeacd8\traces\raw_trace.json
  → Output: C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\55363bdb-3479-4bca-989d-b60b8afeacd8


2026-05-26 22:18:27,655 - [fault_bucketing.py : run : 1178] - INFO - Batch 35/82 processed (1 events): fault_span_skips=0, scaffolding_skips=0, deterministic_assignments=0, zero_candidate_skips=0, llm_classifications=1, tokens=6865/370
2026-05-26 22:18:27,660 - [azure_openai_util.py : _get_or_create_agent : 137] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-26 22:18:31,035 - [fault_bucketing.py : run : 1178] - INFO - Batch 36/82 processed (1 events): fault_span_skips=0, scaffolding_skips=0, deterministic_assignments=0, zero_candidate_skips=0, llm_classifications=1, tokens=8692/318
2026-05-26 22:18:31,040 - [azure_openai_util.py : _get_or_create_agent : 137] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-26 22:18:33,900 - [fault_bucketing.py : run : 1178] - INFO - Batch 37/82 processed (1 events): fault_span_skips=0, scaffolding_skips=0, deterministic_assignments=0, zero_candidate_skip

...

  ✓ SUCCESS: 3 faults extracted, 2,535,482 tokens used

[9/10] Folder: c52aa625-cd95-43e6-9f6f-7ddf1def07ef
  → Input : C:\Users\meemankgupta\Music\Project\infosys\certifier\data\input\21-05-ravi-top10\1960bc89-361a-4fcb-af86-699113f09ec9-langfuse\e8650990-3905-4e56-b458-1ddb84d3972e\fault-bucketing\c52aa625-cd95-43e6-9f6f-7ddf1def07ef\traces\raw_trace.json
  → Output: C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\c52aa625-cd95-43e6-9f6f-7ddf1def07ef


2026-05-27 00:29:00,163 - [fault_bucketing.py : run : 1178] - INFO - Batch 31/84 processed (1 events): fault_span_skips=0, scaffolding_skips=0, deterministic_assignments=0, zero_candidate_skips=0, llm_classifications=1, tokens=6796/396
2026-05-27 00:29:00,167 - [azure_openai_util.py : _get_or_create_agent : 137] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-27 00:29:02,826 - [fault_bucketing.py : _record_fault_detection : 810] - INFO - Fault detection recorded for bucket 'pod-cpu-hog' at 2026-05-11T14:02:30.297Z
2026-05-27 00:29:02,827 - [fault_bucketing.py : run : 1178] - INFO - Batch 32/84 processed (1 events): fault_span_skips=0, scaffolding_skips=0, deterministic_assignments=0, zero_candidate_skips=0, llm_classifications=1, tokens=8479/370
2026-05-27 00:29:02,828 - [fault_bucketing.py : run : 1178] - INFO - Batch 33/84 processed (1 events): fault_span_skips=0, scaffolding_skips=1, deterministic_assignments=0, zero_candidate_skips=

  ✓ SUCCESS: 3 faults extracted, 2,261,714 tokens used

[10/10] Folder: da495936-aa5c-4fb1-a0a8-7cf5e2922222
  → Input : C:\Users\meemankgupta\Music\Project\infosys\certifier\data\input\21-05-ravi-top10\1960bc89-361a-4fcb-af86-699113f09ec9-langfuse\e8650990-3905-4e56-b458-1ddb84d3972e\fault-bucketing\da495936-aa5c-4fb1-a0a8-7cf5e2922222\traces\raw_trace.json
  → Output: C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\da495936-aa5c-4fb1-a0a8-7cf5e2922222


2026-05-27 00:41:29,931 - [fault_bucketing.py : run : 1178] - INFO - Batch 31/81 processed (1 events): fault_span_skips=0, scaffolding_skips=0, deterministic_assignments=0, zero_candidate_skips=0, llm_classifications=1, tokens=6803/364
2026-05-27 00:41:29,937 - [azure_openai_util.py : _get_or_create_agent : 137] - WARNING - No specific client found for model 'gpt-4o_structured'. Using default client.
2026-05-27 00:41:33,002 - [fault_bucketing.py : run : 1178] - INFO - Batch 32/81 processed (1 events): fault_span_skips=0, scaffolding_skips=0, deterministic_assignments=0, zero_candidate_skips=0, llm_classifications=1, tokens=8986/307
2026-05-27 00:41:33,003 - [fault_bucketing.py : run : 1178] - INFO - Batch 33/81 processed (1 events): fault_span_skips=0, scaffolding_skips=1, deterministic_assignments=0, zero_candidate_skips=0, llm_classifications=0, tokens=0/0
2026-05-27 00:41:33,004 - [fault_bucketing.py : run : 1178] - INFO - Batch 34/81 processed (1 events): fault_span_skips=0, scaffo

  ✓ SUCCESS: 3 faults extracted, 2,338,741 tokens used

Done in 2:47:35.443722. Success: 10, Skipped: 0, Failed: 0


## 6. Generate Summary Report

In [6]:
# Create summary DataFrame with explicit columns so it works even when empty
SUMMARY_COLUMNS = ["index", "folder_id", "output_path", "status", "faults_extracted", "total_tokens", "error"]
df_summary = pd.DataFrame(results_summary, columns=SUMMARY_COLUMNS) if results_summary else pd.DataFrame(columns=SUMMARY_COLUMNS)

# Display summary statistics
print("=" * 80)
print("BATCH PROCESSING SUMMARY")
print("=" * 80)
print(f"\nTotal traces processed: {len(df_summary)}")
print(f"Successful: {len(df_summary[df_summary['status'] == 'success'])}")
print(f"Failed: {len(df_summary[df_summary['status'] == 'failed'])}")
print(f"\nTotal faults extracted: {df_summary['faults_extracted'].sum()}")
print(f"Total tokens used: {df_summary['total_tokens'].sum():,}")
if len(df_summary) > 0:
    print(f"Average tokens per trace: {df_summary['total_tokens'].mean():.0f}")

# Display full summary table
print("\n" + "=" * 80)
print("DETAILED RESULTS")
print("=" * 80)
display(df_summary)

# Save summary to CSV
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)
summary_file = OUTPUT_BASE_DIR / "batch_processing_summary.csv"
df_summary.to_csv(summary_file, index=False)
print(f"\n✓ Summary saved to: {summary_file}")

# Display failures if any
failures = df_summary[df_summary['status'] == 'failed']
if len(failures) > 0:
    print(f"\n⚠ {len(failures)} traces failed:")
    for _, row in failures.iterrows():
        print(f"  - [{row['index']}] {row['folder_id']}: {row['error']}")

BATCH PROCESSING SUMMARY

Total traces processed: 10
Successful: 10
Failed: 0

Total faults extracted: 32
Total tokens used: 22,828,388
Average tokens per trace: 2282839

DETAILED RESULTS


,index,folder_id,output_path,status,faults_extracted,total_tokens,error
0,NaN,000098ae-eac9-4042-ac11-546006172858,C:\Users\meemankgupta\Music\Project\infosys\ce...,success,3,2270781,None
1,NaN,55363bdb-3479-4bca-989d-b60b8afeacd8,C:\Users\meemankgupta\Music\Project\infosys\ce...,success,4,2305609,None
2,NaN,71f3c167-5202-4750-9de4-0b4ca44e726b,C:\Users\meemankgupta\Music\Project\infosys\ce...,success,4,1814522,None
3,NaN,a3d3785f-7521-4b73-85d4-e0e5c5a54603,C:\Users\meemankgupta\Music\Project\infosys\ce...,success,3,2245959,None
4,NaN,a599d630-f6a7-4269-b300-c7b87242c1aa,C:\Users\meemankgupta\Music\Project\infosys\ce...,success,3,2340160,None
5,NaN,a9122c3d-4cd3-4a0b-bf99-b8151d7c1170,C:\Users\meemankgupta\Music\Project\infosys\ce...,success,3,2293409,None
6,NaN,ada79287-1eb5-4e6b-8611-cdc21af9e263,C:\Users\meemankgupta\Music\Project\infosys\ce...,success,3,2422011,None
7,NaN,b1bf3fb2-39ad-49b3-abf0-240d7ed2be20,C:\Users\meemankgupta\Music\Project\infosys\ce...,success,3,2535482,None
8,NaN,c52aa625-cd95-43e6-9f6f-7ddf1def07ef,C:\Users\meemankgupta\Music\Project\infosys\ce...,success,3,2261714,None
9,NaN,da495936-aa5c-4fb1-a0a8-7cf5e2922222,C:\Users\meemankgupta\Music\Project\infosys\ce...,success,3,2338741,None



✓ Summary saved to: C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\batch_processing_summary.csv


## 7. Run Full Certification Pipeline (Aggregation → Hypothesis → Report)

In [7]:
from run_full_certification_pipeline import run_pipeline as run_full_pipeline

# ── Configuration ────────────────────────────────────────────────────────────
CERT_AGENT_ID   = "1960bc89-361a-4fcb-af86-699113f09ec9"
CERT_AGENT_NAME = "demoinfra2_base"
CERT_ADVANCED   = True

CERT_METRICS_DIR = OUTPUT_BASE_DIR
CERT_OUTPUT_DIR  = OUTPUT_BASE_DIR / "cert_output"

print(f"Metrics dir : {CERT_METRICS_DIR}")
print(f"Output dir  : {CERT_OUTPUT_DIR}")
print(f"Agent ID    : {CERT_AGENT_ID}")
print(f"Agent name  : {CERT_AGENT_NAME}")
print(f"Advanced    : {CERT_ADVANCED}")
print(f"Metrics dir exists: {CERT_METRICS_DIR.exists()}")

cert_result = await run_full_pipeline(
    metrics_dir=str(CERT_METRICS_DIR),
    output_dir=str(CERT_OUTPUT_DIR),
    agent_id=CERT_AGENT_ID,
    agent_name=CERT_AGENT_NAME,
    advanced_analysis=CERT_ADVANCED,
)

CERT_JSON_PATH = CERT_OUTPUT_DIR / f"certification_report_{CERT_AGENT_ID}.json"
print(f"\n✓ Certification pipeline complete.")
print(f"  Cert JSON: {CERT_JSON_PATH}")

2026-05-27 00:54:19,941 - [azure_openai_util.py : __init__ : 46] - INFO - AzureLLMClient initialized successfully
2026-05-27 00:54:19,942 - [run_full_certification_pipeline.py : run_pipeline : 464] - INFO - ============================================================
2026-05-27 00:54:19,943 - [run_full_certification_pipeline.py : run_pipeline : 465] - INFO - STEP 1: Aggregation
2026-05-27 00:54:19,943 - [run_full_certification_pipeline.py : run_pipeline : 466] - INFO - ============================================================


Metrics dir : C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10
Output dir  : C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\cert_output
Agent ID    : 1960bc89-361a-4fcb-af86-699113f09ec9
Agent name  : demoinfra2_base
Advanced    : True
Metrics dir exists: True


2026-05-27 00:54:20,670 - [aggregation.py : _load_all_docs : 220] - INFO - Loaded 32 documents from C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10
2026-05-27 00:54:20,672 - [aggregation.py : query_runs_by_agent : 232] - INFO - Found 32 documents for agent_id='1960bc89-361a-4fcb-af86-699113f09ec9' in C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10
2026-05-27 00:54:20,673 - [run_full_certification_pipeline.py : run_pipeline : 479] - INFO - Found 32 per-run documents for agent_id='1960bc89-361a-4fcb-af86-699113f09ec9'
2026-05-27 00:54:20,692 - [run_full_certification_pipeline.py : _group_docs_by_category : 178] - WARNING - Could not determine fault_category for doc with fault_name='pod-network-loss6zfpq'; skipping.
2026-05-27 00:54:20,693 - [run_full_certification_pipeline.py : _group_docs_by_category : 178] - WARNING - Could not determine fault_category for doc with fault_name='pod-cpu-hoggx2b8'; skipping.
2026-05-27 


AGGREGATION SUMMARY
  Agent: demoinfra2_base (1960bc89-361a-4fcb-af86-699113f09ec9)
  Total categories: 2
  Total faults tested: 3
  Total runs: 10

  Category: network_fault
    Total runs: 10
    Faults tested: pod-network-loss

  Category: resource_fault
    Total runs: 10
    Faults tested: pod-cpu-hog, pod-memory-hog


2026-05-27 00:55:42,270 - [run_full_certification_pipeline.py : _run_hypothesis_with_gate : 306] - INFO - Per-category min-runs gate failed: Insufficient total runs per category (need 30 each): resource_fault: 10 runs (need 30); network_fault: 10 runs (need 30)
2026-05-27 00:55:42,272 - [run_full_certification_pipeline.py : run_pipeline : 618] - INFO - statistical_hypothesis status: skipped
2026-05-27 00:55:42,274 - [run_full_certification_pipeline.py : run_pipeline : 641] - INFO - Aggregated scorecard written to C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\cert_output\aggregated_scorecard_output_1960bc89-361a-4fcb-af86-699113f09ec9.json
2026-05-27 00:55:42,276 - [run_full_certification_pipeline.py : run_pipeline : 646] - INFO - ============================================================
2026-05-27 00:55:42,277 - [run_full_certification_pipeline.py : run_pipeline : 647] - INFO - STEP 3: Certification
2026-05-27 00:55:42,277 - [run_full_certificati

=== Certification Pipeline Started ===
[pipeline] Phase 1: Ingestion
[pipeline] Phase 2: Computation
[pipeline] Phase 3: Narratives
[narrative-assembler] After scope_narrative: {'input_tokens': 352, 'output_tokens': 189, 'total_tokens': 541}
[narrative-assembler] After key_findings: {'input_tokens': 2374, 'output_tokens': 606, 'total_tokens': 2980}
[narrative-assembler] After qualitative: {'input_tokens': 4207, 'output_tokens': 1127, 'total_tokens': 5334}
[narrative-assembler] After fault_analysis: {'input_tokens': 7530, 'output_tokens': 1716, 'total_tokens': 9246}
[narrative-assembler] After limitations: {'input_tokens': 10208, 'output_tokens': 2774, 'total_tokens': 12982}
[narrative-assembler] After fairness_score: {'input_tokens': 11100, 'output_tokens': 2896, 'total_tokens': 13996}
[narrative-assembler] After recommendations: {'input_tokens': 14160, 'output_tokens': 4344, 'total_tokens': 18504}
[narrative-assembler] Done in 37.8s
[narrative-assembler] Phase 3 tokens: {'input_tokens

2026-05-27 00:56:45,703 - [run_full_certification_pipeline.py : run_pipeline : 667] - INFO - Certification report written to C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\cert_output\certification_report_1960bc89-361a-4fcb-af86-699113f09ec9.json
2026-05-27 00:56:45,709 - [run_full_certification_pipeline.py : run_pipeline : 690] - INFO - ============================================================
2026-05-27 00:56:45,710 - [run_full_certification_pipeline.py : run_pipeline : 691] - INFO - Pipeline Complete
2026-05-27 00:56:45,711 - [run_full_certification_pipeline.py : run_pipeline : 692] - INFO - ============================================================
2026-05-27 00:56:45,714 - [run_full_certification_pipeline.py : run_pipeline : 693] - INFO -   Agent              : demoinfra2_base (1960bc89-361a-4fcb-af86-699113f09ec9)
2026-05-27 00:56:45,715 - [run_full_certification_pipeline.py : run_pipeline : 694] - INFO -   Fault categories   : 2
2026-05-2

=== Pipeline Completed in 63.4s ===
Output: C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\cert_output\certification_report_1960bc89-361a-4fcb-af86-699113f09ec9.json

✓ Certification pipeline complete.
  Cert JSON: C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\cert_output\certification_report_1960bc89-361a-4fcb-af86-699113f09ec9.json


## 8. Generate HTML/PDF Report via Cert Reporter

In [8]:
import sys

# cert_reporter uses relative imports — add its root to sys.path
CERT_REPORTER_DIR = str(BASE_DIR / "cert_reporter")
if CERT_REPORTER_DIR not in sys.path:
    sys.path.insert(0, CERT_REPORTER_DIR)

from pipeline.graph import run_pipeline as run_reporter_pipeline

REPORTER_OUTPUT_DIR = CERT_OUTPUT_DIR / "report"

print(f"Cert JSON  : {CERT_JSON_PATH}")
print(f"Report dir : {REPORTER_OUTPUT_DIR}")
print(f"Cert JSON exists: {CERT_JSON_PATH.exists()}")

reporter_state = run_reporter_pipeline(
    input_path=str(CERT_JSON_PATH),
    output_dir=str(REPORTER_OUTPUT_DIR),
    formats=["html", "pdf"],
)

html_path = reporter_state.get("html_path", "")
pdf_path  = reporter_state.get("pdf_path", "")

print("\n✓ Cert reporter complete.")
if html_path:
    print(f"  HTML → {html_path}")
if pdf_path:
    print(f"  PDF  → {pdf_path}")
for err in reporter_state.get("errors", []):
    print(f"  ⚠ {err}")

Cert JSON  : C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\cert_output\certification_report_1960bc89-361a-4fcb-af86-699113f09ec9.json
Report dir : C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\cert_output\report
Cert JSON exists: True


[2026-05-27 00:56:52 - c:\Users\meemankgupta\AppData\Local\Programs\Python\Python312\Lib\asyncio\base_events.py:1833 - ERROR] Task exception was never retrieved
future: <Task finished name='Task-407' coro=<Connection.run() done, defined at c:\Users\meemankgupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\playwright\_impl\_connection.py:303> exception=NotImplementedError()>
Traceback (most recent call last):
  File "c:\Users\meemankgupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\playwright\_impl\_connection.py", line 310, in run
    await self._transport.connect()
  File "c:\Users\meemankgupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\playwright\_impl\_transport.py", line 133, in connect
    raise exc
  File "c:\Users\meemankgupta\AppData\Local\Programs\Python\Python312\Lib\site-packages\playwright\_impl\_transport.py", line 120, in connect
    self._proc = await asyncio.create_subprocess_exec(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


✓ Cert reporter complete.
  HTML → C:\Users\meemankgupta\Music\Project\infosys\certifier\data\output\26-05-ravi-top10\cert_output\report\cert-e8650990-3905-4e56-b458-1ddb84d3972e.html
  ⚠ PDF render failed: 
